# BART-FaCT: 实验运行 Notebook

本 Notebook 用于运行 BART-FaCT 的各项实验。

**核心思路：** 在已在下游任务上预训练好的 BART-Large-CNN 基础上，添加 HSE / CFA / CPO 三个模块进行消融实验，并与多个 HuggingFace 预训练模型对比。

**使用方法：**
1. 先依次运行 **0. 环境配置** 的所有 Cell
2. 根据需要选择运行下方任意实验 Cell，各实验相互独立

**实验列表：**
- **Exp 1:** 多模型对比实验 (BART-Large-CNN / BART-Base / PEGASUS-arXiv / DistilBART-CNN / BART-FaCT)
- **Exp 2:** 消融实验 (逐个移除 HSE / CFA / CPO)
- **Exp 3:** 幻觉检测与分析
- **Exp 4:** 上下文长度影响实验
- **Exp 5:** 参数敏感性分析 (beam_size / length_penalty / CPO alpha / CFA dim / epochs)
- **Exp 6:** 截断策略对比实验

## 0. 环境配置

依次运行以下 3 个 Cell 完成环境搭建。

In [1]:
# Cell 0-1: 安装依赖
!pip install torch>=2.0.0
!pip install transformers>=4.35.0
!pip install datasets>=2.14.0
!pip install accelerate>=0.24.0
!pip install rouge-score>=0.1.2
!pip install bert-score>=0.3.13
!pip install nltk>=3.8
!pip install matplotlib>=3.7.0
!pip install seaborn>=0.12.0
!pip install pandas>=2.0.0
!pip install numpy>=1.24.0
!pip install scikit-learn>=1.3.0
!pip install tqdm>=4.65.0
!pip install sentencepiece>=0.1.99
!pip install protobuf>=4.21.0
!pip install tensorboard>=2.14.0

In [2]:
# Cell 0-2: 克隆仓库并设置路径
import os, sys

REPO_URL = "https://github.com/zryshuaige/BART-FaCT.git"
REPO_DIR = "BART-FaCT"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

src_path = os.path.abspath(os.path.join(REPO_DIR, "src"))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.chdir(REPO_DIR)
print(f"工作目录: {os.getcwd()}")
print(f"src 路径: {src_path}")
print(f"src 存在: {os.path.exists(src_path)}")

工作目录: /content/BART-FaCT
src 路径: /content/BART-FaCT/src
src 存在: True


In [3]:
# Cell 0-3: 验证环境 & 设置 HuggingFace 端点
# Colab 用 huggingface.co（官方），国内环境用 hf-mirror.com（镜像）
import os

IN_COLAB = 'google.colab' in str(get_ipython()) if hasattr(get_ipython(), '__module__') else False
HF_ENDPOINT = "https://huggingface.co" if IN_COLAB else "https://hf-mirror.com"

os.environ["HF_ENDPOINT"] = HF_ENDPOINT
os.environ["HUGGINGFACE_HUB_TIMEOUT"] = "120"

import torch
import datasets
import transformers
import huggingface_hub

datasets.config.HF_ENDPOINT = HF_ENDPOINT
if hasattr(huggingface_hub, "constants"):
    huggingface_hub.constants.HF_ENDPOINT = HF_ENDPOINT
if hasattr(huggingface_hub, "HF_ENDPOINT"):
    huggingface_hub.HF_ENDPOINT = HF_ENDPOINT
transformers.utils.hub.HF_ENDPOINT = HF_ENDPOINT

from config import get_device, BARTFaCTConfig, ABLATION_CONFIGS, MODEL_CONFIGS
from data_utils import set_seed, _ensure_mirror

# 修补 _ensure_mirror，防止它强制覆盖为 hf-mirror.com
import data_utils as _du
_du._ensure_mirror = lambda: None

device = get_device()
set_seed(42)

print(f"运行环境: {'Colab' if IN_COLAB else '本地'}")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"HF_ENDPOINT: {HF_ENDPOINT}")
print(f"可用模型: {list(MODEL_CONFIGS.keys())}")
print("\n环境验证通过!")

运行环境: Colab
Device: cuda
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB
HF_ENDPOINT: https://huggingface.co
可用模型: ['bart-large-cnn', 'pegasus-cnn_dailymail', 'pegasus-arxiv', 'pegasus-xsum', 'distilbart-cnn-12-6', 'bart-fact-full', 'bart-fact-no-hse', 'bart-fact-no-cfa', 'bart-fact-no-cpo', 'bart-baseline']

环境验证通过!


---

## Exp 1: 多模型对比实验

训练并评估多个摘要模型，对比 BART-Large-CNN / BART-Base / PEGASUS-arXiv / DistilBART-CNN / BART-FaCT 的性能。

可修改 `MODELS` 列表选择要对比的模型，修改 `MAX_SAMPLES` 控制训练数据量。

In [4]:
# Exp 1: 多模型对比实验
from run_experiments import run_experiment_1_model_comparison

#MODELS = ["bart-large-cnn", "bart-base", "pegasus-arxiv", "distilbart-cnn-12-6", "bart-fact-full"]
MODELS = ["bart-fact-full"]
DATASET = "arxiv"
MAX_SAMPLES = 100
NUM_TEST = 10
OUTPUT_DIR = "./results/exp1_model_comparison"

exp1_results, exp1_predictions = run_experiment_1_model_comparison(
    dataset_name=DATASET,
    max_samples=MAX_SAMPLES,
    num_test=NUM_TEST,
    models=MODELS,
    output_dir=OUTPUT_DIR,
)

print("\n" + "="*60)
print("Exp 1 结果摘要:")
print("="*60)
for model_name, result in exp1_results.items():
    if isinstance(result, dict) and "rouge" in result:
        r = result["rouge"]
        print(f"  {model_name}: R1={r['rouge1']['fmeasure']:.4f}  R2={r['rouge2']['fmeasure']:.4f}  RL={r['rougeL']['fmeasure']:.4f}")
    else:
        print(f"  {model_name}: {result}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

/content/BART-FaCT/src/models/hierarchical_structure.py:147: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.sentence_transformer = nn.TransformerEncoder(


Tokenizing arxiv:   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenizing arxiv:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing arxiv:   0%|          | 0/6436 [00:00<?, ? examples/s]

ERROR:run_experiments:Training failed for bart-fact-full: Seq2SeqTrainer.__init__() got an unexpected keyword argument 'tokenizer'



Exp 1 结果摘要:


---

## Exp 2: 消融实验

逐个移除 BART-FaCT 的创新模块 (HSE / CFA / CPO)，验证各模块的贡献。

| 配置 | HSE | CFA | CPO |
|------|-----|------|-----|
| BART (Baseline) | ✗ | ✗ | ✗ |
| w/o HSE | ✗ | ✓ | ✓ |
| w/o CFA | ✓ | ✗ | ✓ |
| w/o CPO | ✓ | ✓ | ✗ |
| **BART-FaCT (Full)** | **✓** | **✓** | **✓** |

In [ ]:
# Exp 2-a: 运行全部消融实验
from run_experiments import run_experiment_2_ablation

DATASET = "arxiv"
MAX_SAMPLES = 1000
NUM_TEST = 100
OUTPUT_DIR = "./results/exp2_ablation"

exp2_results = run_experiment_2_ablation(
    dataset_name=DATASET,
    max_samples=MAX_SAMPLES,
    num_test=NUM_TEST,
    output_dir=OUTPUT_DIR,
)

print("\n" + "="*60)
print("Exp 2 消融实验结果:")
print("="*60)
print(f"{'配置':<25} {'HSE':>5} {'CFA':>5} {'CPO':>5} {'R1':>8} {'R2':>8} {'RL':>8}")
print("-"*70)
for name, result in exp2_results.items():
    if isinstance(result, dict) and "error" not in result:
        r = result.get("eval_results", {}).get("rouge", {})
        print(f"{name:<25} {str(result.get('use_hse','-')):>5} {str(result.get('use_cfa','-')):>5} "
              f"{str(result.get('use_cpo','-')):>5} {r.get('rouge1',{}).get('fmeasure',0):>8.4f} "
              f"{r.get('rouge2',{}).get('fmeasure',0):>8.4f} {r.get('rougeL',{}).get('fmeasure',0):>8.4f}")
    else:
        print(f"{name:<25} 失败: {result.get('error', 'unknown')}")

In [ ]:
# Exp 2-b: 运行单个消融实验（可选）
from ablation import run_single_ablation

ABLATION_NAME = "bart_fact_no_hse"

result = run_single_ablation(
    ablation_name=ABLATION_NAME,
    dataset_name="arxiv",
    max_samples=1000,
    num_test=100,
    output_dir="./results/exp2_ablation",
)

print(f"\n消融实验 {ABLATION_NAME} 完成")
if "eval_results" in result:
    r = result["eval_results"]["rouge"]
    print(f"  R1={r['rouge1']['fmeasure']:.4f}  R2={r['rouge2']['fmeasure']:.4f}  RL={r['rougeL']['fmeasure']:.4f}")

---

## Exp 3: 幻觉检测与分析

使用 NLI 模型检测生成摘要中的幻觉内容，分析幻觉类型分布。

需要先运行 Exp 1 或 Exp 2 生成预测结果，或直接加载已有结果文件。

In [ ]:
# Exp 3: 幻觉检测与分析
from run_experiments import run_experiment_3_hallucination
from data_utils import load_arxiv_dataset

DATASET = "arxiv"
NUM_TEST = 100
OUTPUT_DIR = "./results/exp3_hallucination"

if 'exp1_predictions' in globals() and exp1_predictions:
    ds = load_arxiv_dataset()
    test_key = "test" if "test" in ds else "validation"
    source_texts = [sample["article"] for sample in ds[test_key].select(range(NUM_TEST))]

    exp3_results = run_experiment_3_hallucination(
        predictions_dict=exp1_predictions,
        source_texts=source_texts,
        output_dir=OUTPUT_DIR,
    )

    print("\n" + "="*60)
    print("Exp 3 幻觉检测结果:")
    print("="*60)
    for model_name, result in exp3_results.items():
        nli = result.get("nli_metrics", {})
        print(f"  {model_name}:")
        print(f"    事实率: {nli.get('factuality_rate', 'N/A')}")
        print(f"    幻觉率: {nli.get('hallucination_rate', 'N/A')}")
        print(f"    矛盾率: {nli.get('mean_contradiction', 'N/A')}")
else:
    print("请先运行 Exp 1 生成预测结果，或手动加载 predictions.json 文件。")

---

## Exp 4: 上下文长度影响实验

测试 BART-FaCT 在不同上下文长度下的性能变化。

上下文长度: 256 / 512 / 768 / 1024

In [ ]:
# Exp 4: 上下文长度影响实验
from run_experiments import run_experiment_4_context_length

MODEL_NAME = "bart-fact-full"
DATASET = "arxiv"
CONTEXT_LENGTHS = [256, 512, 768, 1024]
MAX_SAMPLES = 1000
NUM_TEST = 100
OUTPUT_DIR = "./results/exp4_context_length"

exp4_results = run_experiment_4_context_length(
    model_name=MODEL_NAME,
    dataset_name=DATASET,
    context_lengths=CONTEXT_LENGTHS,
    max_samples=MAX_SAMPLES,
    num_test=NUM_TEST,
    output_dir=OUTPUT_DIR,
)

print("\n" + "="*60)
print("Exp 4 上下文长度影响结果:")
print("="*60)
for ctx_len, result in exp4_results.items():
    if isinstance(result, dict) and "error" not in result:
        r = result.get("rouge", {})
        print(f"  ctx={ctx_len}: R1={r.get('rouge1',{}).get('fmeasure',0):.4f}  "
              f"R2={r.get('rouge2',{}).get('fmeasure',0):.4f}  RL={r.get('rougeL',{}).get('fmeasure',0):.4f}")
    else:
        print(f"  ctx={ctx_len}: 失败")

---

## Exp 5: 参数敏感性分析

分析各超参数对模型性能的影响。

可单独运行各子实验，也可一次性运行全部。

In [ ]:
# Exp 5-a: Beam Size 敏感性
from sensitivity import sensitivity_beam_size

beam_results = sensitivity_beam_size(
    model_name="bart-fact-full",
    dataset_name="arxiv",
    beam_sizes=[1, 2, 4, 6, 8],
    num_test=100,
    output_dir="./results/exp5_sensitivity",
)

print("\nBeam Size 敏感性结果:")
for beam, result in beam_results.items():
    r = result.get("rouge", {})
    print(f"  beam={beam}: R1={r.get('rouge1',{}).get('fmeasure',0):.4f}  "
          f"R2={r.get('rouge2',{}).get('fmeasure',0):.4f}  RL={r.get('rougeL',{}).get('fmeasure',0):.4f}")

In [ ]:
# Exp 5-b: Length Penalty 敏感性
from sensitivity import sensitivity_length_penalty

lp_results = sensitivity_length_penalty(
    model_name="bart-fact-full",
    dataset_name="arxiv",
    length_penalties=[0.6, 1.0, 1.5, 2.0, 2.5],
    num_test=200,
    output_dir="./results/exp5_sensitivity",
)

print("\nLength Penalty 敏感性结果:")
for lp, result in lp_results.items():
    r = result.get("rouge", {})
    print(f"  lp={lp}: R1={r.get('rouge1',{}).get('fmeasure',0):.4f}  "
          f"R2={r.get('rouge2',{}).get('fmeasure',0):.4f}  RL={r.get('rougeL',{}).get('fmeasure',0):.4f}")

In [ ]:
# Exp 5-c: CPO Alpha 敏感性
from sensitivity import sensitivity_cpo_alpha

cpo_results = sensitivity_cpo_alpha(
    model_name="bart-fact-full",
    dataset_name="arxiv",
    alphas=[0.01, 0.05, 0.1, 0.2, 0.5],
    max_samples=1000,
    num_test=100,
    output_dir="./results/exp5_sensitivity",
)

print("\nCPO Alpha 敏感性结果:")
for alpha, result in cpo_results.items():
    if "error" not in result:
        r = result.get("eval_results", {}).get("rouge", {})
        print(f"  alpha={alpha}: R1={r.get('rouge1',{}).get('fmeasure',0):.4f}  "
              f"R2={r.get('rouge2',{}).get('fmeasure',0):.4f}  RL={r.get('rougeL',{}).get('fmeasure',0):.4f}")
    else:
        print(f"  alpha={alpha}: 失败 - {result['error']}")

In [ ]:
# Exp 5-d: CFA Hidden Dim 敏感性
from sensitivity import sensitivity_cfa_dim

cfa_results = sensitivity_cfa_dim(
    model_name="bart-fact-full",
    dataset_name="arxiv",
    dims=[32, 64, 128, 256],
    max_samples=1000,
    num_test=100,
    output_dir="./results/exp5_sensitivity",
)

print("\nCFA Hidden Dim 敏感性结果:")
for dim, result in cfa_results.items():
    if "error" not in result:
        r = result.get("eval_results", {}).get("rouge", {})
        print(f"  dim={dim}: R1={r.get('rouge1',{}).get('fmeasure',0):.4f}  "
              f"R2={r.get('rouge2',{}).get('fmeasure',0):.4f}  RL={r.get('rougeL',{}).get('fmeasure',0):.4f}")
    else:
        print(f"  dim={dim}: 失败 - {result['error']}")

In [ ]:
# Exp 5-e: Epochs 敏感性
from sensitivity import sensitivity_epochs

epochs_results = sensitivity_epochs(
    model_name="bart-fact-full",
    dataset_name="arxiv",
    epochs=[1, 2, 3, 5],
    max_samples=1000,
    num_test=100,
    output_dir="./results/exp5_sensitivity",
)

print("\nEpochs 敏感性结果:")
for num_ep, result in epochs_results.items():
    if "error" not in result:
        r = result.get("eval_results", {}).get("rouge", {})
        print(f"  epochs={num_ep}: R1={r.get('rouge1',{}).get('fmeasure',0):.4f}  "
              f"R2={r.get('rouge2',{}).get('fmeasure',0):.4f}  RL={r.get('rougeL',{}).get('fmeasure',0):.4f}")
    else:
        print(f"  epochs={num_ep}: 失败 - {result['error']}")

In [ ]:
# Exp 5-f: 运行全部敏感性分析（耗时较长）
from sensitivity import run_all_sensitivity

all_sensitivity_results = run_all_sensitivity(
    model_name="bart-fact-full",
    dataset_name="arxiv",
    max_samples=1000,
    num_test=100,
    output_dir="./results/exp5_sensitivity",
)

print("\n全部敏感性分析完成!")
for analysis_name, result in all_sensitivity_results.items():
    print(f"  {analysis_name}: done")

---

## Exp 6: 截断策略对比实验

对比不同截断策略对长文档摘要性能的影响：
- **head_only**: 只取前 N 个 token
- **tail_only**: 只取后 N 个 token
- **head_tail_mixed**: 前半 + 后半混合

In [ ]:
# Exp 6: 截断策略对比
from sensitivity import sensitivity_truncation_strategy

truncation_results = sensitivity_truncation_strategy(
    model_name="bart-fact-full",
    dataset_name="arxiv",
    strategies=["head_only", "tail_only", "head_tail_mixed"],
    max_input_length=1024,
    num_test=100,
    output_dir="./results/exp6_truncation",
)

print("\n" + "="*60)
print("Exp 6 截断策略对比结果:")
print("="*60)
for strategy, result in truncation_results.items():
    r = result.get("rouge", {})
    print(f"  {strategy:20s}: R1={r.get('rouge1',{}).get('fmeasure',0):.4f}  "
          f"R2={r.get('rouge2',{}).get('fmeasure',0):.4f}  RL={r.get('rougeL',{}).get('fmeasure',0):.4f}")

---

## 快速测试

用少量数据快速验证代码流程是否正确，不反映真实性能。

In [ ]:
# 快速测试: 少量数据验证流程
from run_experiments import run_experiment_1_model_comparison

quick_results, _ = run_experiment_1_model_comparison(
    dataset_name="arxiv",
    max_samples=100,
    num_test=10,
    models=["bart-fact-full"],
    output_dir="./results/quick_test",
)

for model_name, result in quick_results.items():
    if isinstance(result, dict) and "rouge" in result:
        r = result["rouge"]
        print(f"  {model_name}: R1={r['rouge1']['fmeasure']:.4f}  R2={r['rouge2']['fmeasure']:.4f}  RL={r['rougeL']['fmeasure']:.4f}")

print("\n快速测试完成!")

---

## 完整流水线

一次性运行全部实验（Exp1~Exp5），生成完整结果和图表。

In [ ]:
# 完整流水线: 运行全部实验
from run_experiments import run_full_pipeline

results_dir = run_full_pipeline(
    dataset_name="arxiv",
    max_samples=1000,
    num_test=100,
    output_dir="./results",
    models=["bart-large-cnn", "bart-base", "pegasus-arxiv", "distilbart-cnn-12-6", "bart-fact-full"],
    context_lengths=[256, 512, 768, 1024],
)

print(f"\n全部实验完成! 结果保存在: {results_dir}")